In [32]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json

In [33]:
EXPERIMENTS_DIR = Path('./experiments')

In [34]:
def macro_f1(confusion_matrix):
    num_classes = confusion_matrix.shape[0]
    
    precision = np.zeros(num_classes)
    recall = np.zeros(num_classes)
    
    for i in range(num_classes):
        TP = confusion_matrix[i, i]
        FP = confusion_matrix[:, i].sum() - TP
        FN = confusion_matrix[i, :].sum() - TP
        precision[i] = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall[i] = TP / (TP + FN) if (TP + FN) > 0 else 0
    
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
    
    macro_f1 = np.mean(f1_scores)
    return macro_f1

In [35]:
exp_dir = Path(EXPERIMENTS_DIR) / 'HyperparameterSearch5FoldUnivariate'

best_params = pd.DataFrame()
for ds_dir in sorted(exp_dir.iterdir()):
    if ds_dir.name in ['current', 'curr_log.txt']:
        continue
    try:
        df = pd.DataFrame()
        for proto_len_dir in sorted(ds_dir.iterdir(), key=lambda x: int(x.name.split('-')[-1])):
            proto_len = proto_len_dir.name.split('-')[-1]        
            df[proto_len] = None

            accus = []
            f1s = []
            for fold_dir in proto_len_dir.iterdir():
                fold_idx = fold_dir.name.split('-')[-1]

                curr_fold_stats = json.load((fold_dir / 'stats.json').open())
                
                accu = [x['value'] for x in curr_fold_stats['accu_val']][-1]
                accus.append(accu)
                
                np.loadtxt(fold_dir / 'confusion_matrix.txt')
                f1s.append(macro_f1(np.loadtxt(fold_dir / 'confusion_matrix.txt')))
            
            df.at['accu', proto_len] = np.mean(accus)
            df.at['f1', proto_len] = np.mean(f1s)
        
        df = df.astype(float)
        # plt.figure(figsize=(6, 2))
        # sns.heatmap(df, annot=True, fmt=".2f", cmap='viridis')
        # plt.grid(False)
        # plt.title(ds_dir.name)
        # plt.show()
        
        best_params.at[ds_dir.name, 'proto_len'] = int(df.columns[df.loc['f1'].argmax()])
    except Exception as e:
        print(f'Error processing {ds_dir.name}: {e}')
        raise
    # break
    
best_params['epochs'] = 200

In [30]:
best_params = best_params[['epochs', 'proto_len']].astype(int)
best_params.index.name = 'dataset'
best_params.to_csv('./best_params_uni.csv')